# CFPB Complaints: Intent and Entity Extraction with Judge Evaluation

This notebook adapts the extraction workflow to CFPB consumer complaints and evaluates predictions using an LLM judge.
This time, we have chosen a more complex dataset that cannot be evaluated with a yes or no answer.
In this case using a LLM-as-Judge result the most effective approach to evaluate.



## Setup
Load libraries for data loading, prompt rendering, model calls, and evaluation.

In [10]:
import json
import os
import pandas as pd
import jinja2
from openai import OpenAI

## Load CFPB Dataset
Read the dataset from a local file path (no internet download).

In [11]:
# Local dataset path priority: CFPB_DATASET_PATH env var, then common local filenames
local_candidates = [
    "data/complaints.csv.zip",
    "data/complaints.csv",
    "complaints.csv.zip",
    "complaints.csv",
]

dataset_path = os.getenv("CFPB_DATASET_PATH")
if not dataset_path:
    dataset_path = next((p for p in local_candidates if os.path.exists(p)), None)

if not dataset_path:
    raise FileNotFoundError(
        "No local CFPB dataset found. Set CFPB_DATASET_PATH or place a file at one of: "
        + ", ".join(local_candidates)
    )

# Read only a bounded slice from local file (fast and sufficient for notebook experiments)
max_rows_to_read = 5000
compression = "zip" if str(dataset_path).endswith(".zip") else None
required_cols = ["Consumer complaint narrative", "Product", "Issue"]
raw_df = pd.read_csv(
    dataset_path,
    compression=compression,
    usecols=required_cols,
    nrows=max_rows_to_read,
    low_memory=False,
)

col_map = {
    "Consumer complaint narrative": "narrative",
    "Product": "product",
    "Issue": "issue",
}
df = raw_df.rename(columns=col_map)
df = df.dropna(subset=["narrative", "product", "issue"]).copy()
df["narrative"] = df["narrative"].astype(str).str.strip()
df = df[df["narrative"] != ""].reset_index(drop=True)

print(f"Loaded local dataset from: {dataset_path}")
print(f"Rows read from file: {len(raw_df)} (cap={max_rows_to_read})")
print(f"Rows with narrative + product + issue: {len(df)}")

Loaded local dataset from: data/complaints.csv.zip
Rows read from file: 5000 (cap=5000)
Rows with narrative + product + issue: 1539


## Sample + Analyze
Sample a compact evaluation slice and inspect label distribution.

In [12]:
n_samples = 12
sampled_df = df.sample(n=min(n_samples, len(df)), random_state=42).reset_index(drop=True)

print("Top products")
display(df["product"].value_counts().head(10).rename_axis("product").reset_index(name="count"))

print("Top issues")
display(df["issue"].value_counts().head(10).rename_axis("issue").reset_index(name="count"))

print(f"Sampled rows: {len(sampled_df)}")

Top products


,product,count
0,Credit reporting or other personal consumer re...,578
1,"Credit reporting, credit repair services, or o...",409
2,Debt collection,182
3,Credit card or prepaid card,67
4,Mortgage,66
5,Checking or savings account,63
6,"Money transfer, virtual currency, or money ser...",52
7,Credit card,42
8,Student loan,26
9,Vehicle loan or lease,21


Top issues


,issue,count
0,Incorrect information on your report,489
1,Improper use of your report,238
2,Problem with a company's investigation into an...,127
3,Problem with a credit reporting company's inve...,125
4,Attempts to collect debt not owed,92
5,Managing an account,39
6,Written notification about debt,33
7,Trouble during payment process,32
8,Other transaction problem,23
9,Problem with a purchase shown on your statement,21


Sampled rows: 12


## Few-Shot Construction
Build few-shot examples from frequent product/issue pairs in the dataset.

In [13]:
pair_counts = (
    df.groupby(["product", "issue"], as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(4)
)

few_shot_examples = []
for _, row in pair_counts.iterrows():
    prod = row["product"]
    issue = row["issue"]
    ex = df[(df["product"] == prod) & (df["issue"] == issue)].iloc[0]
    narrative = " ".join(str(ex["narrative"]).split())[:380]
    few_shot_examples.append({
        "product": str(prod),
        "issue": str(issue),
        "narrative": narrative,
    })

few_shots_text = "\n\n".join(
    [
        f"Example {i}:\nNarrative: {ex['narrative']}\nOutput: {json.dumps({'product': ex['product'], 'issue': ex['issue']})}"
        for i, ex in enumerate(few_shot_examples, 1)
    ]
)

print(f"Few-shot examples selected: {len(few_shot_examples)}")
print(few_shots_text)

Few-shot examples selected: 4
Example 1:
Narrative: Kindly address this issue on my credit report. I assert that this account is not mine and believe it to be fraudulent. I urge you to correct this mistake and have provided supporting documents for verification.
Output: {"product": "Credit reporting or other personal consumer reports", "issue": "Incorrect information on your report"}

Example 2:
Narrative: These are not my accounts.
Output: {"product": "Credit reporting, credit repair services, or other personal consumer reports", "issue": "Incorrect information on your report"}

Example 3:
Narrative: Urgent : Disputed Inquiries on Credit Report - Request for Validation Dear TransUnion and Consumer Financial Protection Bureau ( CFPB ), I am writing to express my dissatisfaction with the handling of my recent credit file dispute by TransUnion and to bring to your attention unauthorized inquiries on my credit report. I have recently submitted a dispute regarding inquiries on m
Output: {"

## Prompt Rendering

In [14]:
prompt_env = jinja2.Environment(
    loader=jinja2.FileSystemLoader("prompts"),
    autoescape=False,
)

extract_template = prompt_env.get_template("cfpb_extract_prompt.j2")
review_template = prompt_env.get_template("cfpb_review_prompt.j2")
judge_template = prompt_env.get_template("cfpb_judge_prompt.j2")

records = sampled_df.to_dict(orient="records")

direct_prompts = [
    extract_template.render(narrative=r["narrative"], few_shots=few_shots_text)
    for r in records
]

## LLM Helpers

In [15]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set")

client = OpenAI()

def call_llm_json(prompts, model="gpt-4.1-mini"):
    outputs = []
    for prompt in prompts:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"},
        )
        outputs.append(resp.choices[0].message.content)
    return outputs

def parse_prediction(text):
    data = json.loads(text)
    return {
        "product": str(data.get("product", "")).strip(),
        "issue": str(data.get("issue", "")).strip(),
    }

## Approach 1: Direct Extraction

In [16]:
direct_responses = call_llm_json(direct_prompts)
direct_preds = [parse_prediction(r) for r in direct_responses]

## Approach 2: Separate Review Prompt

In [17]:
review_prompts = [
    review_template.render(
        narrative=r["narrative"],
        extraction=json.dumps(pred),
        few_shots=few_shots_text,
    )
    for r, pred in zip(records, direct_preds)
]
review_responses = call_llm_json(review_prompts)
review_preds = [parse_prediction(r) for r in review_responses]

## Judge Evaluation

In [18]:
def judge_approach(preds, approach_name, model="gpt-4o"):
    prompts = [
        judge_template.render(
            narrative=r["narrative"],
            gold_product=r["product"],
            gold_issue=r["issue"],
            pred_product=p["product"],
            pred_issue=p["issue"],
        )
        for r, p in zip(records, preds)
    ]
    judge_responses = call_llm_json(prompts, model=model)

    rows = []
    for r, p, jr in zip(records, preds, judge_responses):
        jd = json.loads(jr)
        rows.append({
            "approach": approach_name,
            "gold_product": r["product"],
            "gold_issue": r["issue"],
            "pred_product": p["product"],
            "pred_issue": p["issue"],
            "product_match": int(jd.get("product_match", 0)),
            "issue_match": int(jd.get("issue_match", 0)),
            "score": int(jd.get("score", 0)),
            "rating": float(jd.get("rating", 0.0)),
            "verdict": str(jd.get("verdict", "incorrect")),
            "reason": str(jd.get("reason", "")),
        })

    judged_df = pd.DataFrame(rows)
    summary = {
        "approach": approach_name,
        "avg_rating": judged_df["rating"].mean(),
        "avg_score": judged_df["score"].mean(),
        "product_match_rate": judged_df["product_match"].mean(),
        "issue_match_rate": judged_df["issue_match"].mean(),
        "full_correct_rate": (judged_df["verdict"] == "correct").mean(),
    }
    return judged_df, summary

direct_judged, direct_summary = judge_approach(direct_preds, "Direct extraction")
review_judged, review_summary = judge_approach(review_preds, "Separate review prompt")

## Comparison

In [19]:
comparison = pd.DataFrame([direct_summary, review_summary]).sort_values("avg_rating", ascending=False)
display(comparison)

print("Sample judged rows (direct)")
display(direct_judged[["gold_product", "gold_issue", "pred_product", "pred_issue", "score", "rating", "verdict"]].head(5))

,approach,avg_rating,avg_score,product_match_rate,issue_match_rate,full_correct_rate
0,Direct extraction,6.25,1.25,0.833333,0.416667,0.416667
1,Separate review prompt,6.25,1.25,0.750000,0.500000,0.500000


Sample judged rows (direct)


,gold_product,gold_issue,pred_product,pred_issue,score,rating,verdict
0,Credit reporting or other personal consumer re...,Improper use of your report,Credit reporting or other personal consumer re...,Incorrect information on your report,1,5.0,partially_correct
1,Credit reporting or other personal consumer re...,Improper use of your report,Credit reporting or other personal consumer re...,Incorrect information on your report,2,10.0,correct
2,"Credit reporting, credit repair services, or o...",Improper use of your report,Credit reporting or other personal consumer re...,Incorrect information on your report,1,5.0,partially_correct
3,Credit reporting or other personal consumer re...,Incorrect information on your report,Credit reporting or other personal consumer re...,Incorrect information on your report,2,10.0,correct
4,"Payday loan, title loan, or personal loan",Getting a line of credit,Consumer loan,Loan was opened without my consent,0,0.0,incorrect
